# 06 Metrics - Taxonomy-aware Ground Truth

Notebook này dùng để đánh giá 2 mô hình:

- **Without Taxonomy**: `hybrid_no_taxonomy_score`
- **With Taxonomy**: `hybrid_taxonomy_score`

Nhãn `relevant` được xây dựng theo rule phù hợp với đề tài taxonomy:

- `direct_match = 1` nếu skill trùng trực tiếp đủ mạnh.
- `taxonomy_match = 1` nếu skill overlap thấp nhưng cùng nhóm taxonomy rõ ràng.
- `relevant = direct_match OR taxonomy_match`.

Lưu ý quan trọng: để Precision@10/20 và Recall@10/20 có ý nghĩa, mỗi candidate nên có ít nhất 20 jobs trong file ranking.


In [11]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding")
DATA_DIR = BASE_DIR / "data_outputs"

RANKING_PARQUET = DATA_DIR / "13_candidate_job_hybrid_ranking.parquet"
RANKING_EXCEL = DATA_DIR / "13_candidate_job_hybrid_ranking.xlsx"

OUTPUT_CANDIDATE_METRICS = DATA_DIR / "15_candidate_level_metrics.xlsx"
OUTPUT_SUMMARY = DATA_DIR / "16_metrics_summary.xlsx"

print("RANKING_PARQUET exists:", RANKING_PARQUET.exists(), RANKING_PARQUET)
print("RANKING_EXCEL exists:", RANKING_EXCEL.exists(), RANKING_EXCEL)


RANKING_PARQUET exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.parquet
RANKING_EXCEL exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.xlsx


In [12]:
# ===== READ RANKING FILE =====
if RANKING_PARQUET.exists():
    df = pd.read_parquet(RANKING_PARQUET)
elif RANKING_EXCEL.exists():
    df = pd.read_excel(RANKING_EXCEL)
else:
    raise FileNotFoundError("Không tìm thấy 13_candidate_job_hybrid_ranking.parquet hoặc .xlsx")

print("df:", df.shape)
print("Candidates:", df["candidate_id"].nunique())
print("Jobs:", df["job_id"].nunique())
print("Min jobs per candidate:", df.groupby("candidate_id")["job_id"].count().min())
print("Max jobs per candidate:", df.groupby("candidate_id")["job_id"].count().max())

df.head()


df: (100, 20)
Candidates: 20
Jobs: 29
Min jobs per candidate: 5
Max jobs per candidate: 5


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,job_title,match_explanation,semantic_similarity,semantic_rank,semantic_score_norm,baseline_score_norm,hybrid_no_taxonomy_score,hybrid_taxonomy_score,hybrid_score,rank_taxonomy,rank_no_taxonomy,hybrid_rank,score_diff,rank_diff
0,C001,J001,1.0000,1.0,1,1.0000,Java Backend Developer,skill overlap=1.0; group similarity=1.0; same ...,0.897339,1,0.948670,1.0000,1.0000,1.00000,1.00000,1,1,1,0.00000,0
1,C001,J002,0.0000,1.0,1,0.3500,React Frontend Developer,group similarity=1.0; same dominant group,0.840461,17,0.920231,0.3500,0.0000,0.50000,0.50000,2,4,2,0.50000,2
2,C001,J003,0.0000,1.0,1,0.3500,Fullstack Web Developer,group similarity=1.0; same dominant group,0.851088,11,0.925544,0.3500,0.0000,0.50000,0.50000,3,5,3,0.50000,2
3,C001,J025,0.3333,0.5,1,0.4417,Database Developer,skill overlap=0.3333; group similarity=0.5; sa...,0.873763,3,0.936882,0.4417,0.3333,0.49165,0.49165,4,2,4,0.15835,-2
4,C001,J027,0.2000,0.5,1,0.3550,.NET Backend Developer,skill overlap=0.2; group similarity=0.5; same ...,0.867865,7,0.933932,0.3550,0.2000,0.42500,0.42500,5,3,5,0.22500,-2


In [13]:
# ===== CHECK REQUIRED COLUMNS =====
required_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

numeric_cols = [
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df.head()


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,job_title,match_explanation,semantic_similarity,semantic_rank,semantic_score_norm,baseline_score_norm,hybrid_no_taxonomy_score,hybrid_taxonomy_score,hybrid_score,rank_taxonomy,rank_no_taxonomy,hybrid_rank,score_diff,rank_diff
0,C001,J001,1.0000,1.0,1,1.0000,Java Backend Developer,skill overlap=1.0; group similarity=1.0; same ...,0.897339,1,0.948670,1.0000,1.0000,1.00000,1.00000,1,1,1,0.00000,0
1,C001,J002,0.0000,1.0,1,0.3500,React Frontend Developer,group similarity=1.0; same dominant group,0.840461,17,0.920231,0.3500,0.0000,0.50000,0.50000,2,4,2,0.50000,2
2,C001,J003,0.0000,1.0,1,0.3500,Fullstack Web Developer,group similarity=1.0; same dominant group,0.851088,11,0.925544,0.3500,0.0000,0.50000,0.50000,3,5,3,0.50000,2
3,C001,J025,0.3333,0.5,1,0.4417,Database Developer,skill overlap=0.3333; group similarity=0.5; sa...,0.873763,3,0.936882,0.4417,0.3333,0.49165,0.49165,4,2,4,0.15835,-2
4,C001,J027,0.2000,0.5,1,0.3550,.NET Backend Developer,skill overlap=0.2; group similarity=0.5; same ...,0.867865,7,0.933932,0.3550,0.2000,0.42500,0.42500,5,3,5,0.22500,-2


In [14]:
# ===== TAXONOMY-AWARE GROUND TRUTH =====
# Có thể chỉnh threshold nếu cần, nhưng đây là rule dễ giải thích trong báo cáo:
# - Direct match: skill overlap đủ cao
# - Taxonomy match: skill overlap chưa cao nhưng cùng nhóm taxonomy rõ ràng

DIRECT_SKILL_THRESHOLD = 0.50
TAXONOMY_GROUP_THRESHOLD = 0.50

df["direct_match"] = (
    df["skill_overlap_score"] >= DIRECT_SKILL_THRESHOLD
).astype(int)

df["taxonomy_match"] = (
    (df["skill_overlap_score"] < DIRECT_SKILL_THRESHOLD)
    & (df["group_similarity_score"] >= TAXONOMY_GROUP_THRESHOLD)
    & (df["dominant_group_score"] >= 1)
).astype(int)

df["relevant"] = (
    (df["direct_match"] == 1)
    | (df["taxonomy_match"] == 1)
).astype(int)

# ===== TAXONOMY-MATCH SUBSET =====
# Nhóm này dùng để đánh giá riêng các case mà taxonomy có tác dụng:
# skill không trùng trực tiếp nhiều, nhưng cùng nhóm taxonomy rõ ràng.

taxonomy_eval_df = df[
    (df["taxonomy_match"] == 1)
].copy()

print("taxonomy_eval_df:", taxonomy_eval_df.shape)
print("Candidates:", taxonomy_eval_df["candidate_id"].nunique())
print("Jobs:", taxonomy_eval_df["job_id"].nunique())

taxonomy_eval_df[[
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
    "relevant",
]].head(20)

print("Relevant distribution:")
print(df["relevant"].value_counts(dropna=False))

print("\nMatch type summary:")
print("Direct matches:", int(df["direct_match"].sum()))
print("Taxonomy matches:", int(df["taxonomy_match"].sum()))
print("Total relevant:", int(df["relevant"].sum()))

df[[
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "direct_match",
    "taxonomy_match",
    "relevant",
]].head(20)


taxonomy_eval_df: (10, 23)
Candidates: 3
Jobs: 9
Relevant distribution:
relevant
1    72
0    28
Name: count, dtype: int64

Match type summary:
Direct matches: 62
Taxonomy matches: 10
Total relevant: 72


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,direct_match,taxonomy_match,relevant
0,C001,J001,1.0000,1.0,1,1,0,1
1,C001,J002,0.0000,1.0,1,0,1,1
2,C001,J003,0.0000,1.0,1,0,1,1
3,C001,J025,0.3333,0.5,1,0,1,1
4,C001,J027,0.2000,0.5,1,0,1,1
5,C002,J012,1.0000,1.0,1,1,0,1
6,C002,J021,1.0000,1.0,1,1,0,1
7,C002,J028,0.6667,1.0,1,1,0,1
8,C002,J002,0.5000,1.0,1,1,0,1
9,C002,J003,0.5000,1.0,1,1,0,1


In [15]:
# ===== SANITY CHECK =====
job_counts = df.groupby("candidate_id")["job_id"].count()
relevant_counts = df.groupby("candidate_id")["relevant"].sum()

summary_check = pd.DataFrame({
    "n_jobs": job_counts,
    "n_relevant": relevant_counts,
}).reset_index()

print(summary_check.describe())

# Cảnh báo nếu mỗi candidate có ít hơn 20 jobs
if summary_check["n_jobs"].min() < 20:
    print("WARNING: Có candidate có ít hơn 20 jobs. Metrics @20 vẫn tính được, nhưng chưa thật sự đẹp về ý nghĩa.")

summary_check.head(30)


       n_jobs  n_relevant
count    20.0   20.000000
mean      5.0    3.600000
std       0.0    1.930367
min       5.0    0.000000
25%       5.0    2.000000
50%       5.0    5.000000
75%       5.0    5.000000
max       5.0    5.000000


,candidate_id,n_jobs,n_relevant
0,C001,5,5
1,C002,5,5
2,C003,5,5
3,C004,5,3
4,C005,5,0
5,C006,5,0
6,C007,5,2
7,C008,5,2
8,C009,5,5
9,C010,5,5


In [16]:
# ===== METRIC FUNCTIONS =====
K_VALUES = [10, 20]

def precision_at_k(group, rank_col, k):
    top_k = group.sort_values(rank_col).head(k)
    if len(top_k) == 0:
        return 0.0
    return top_k["relevant"].sum() / k


def recall_at_k(group, rank_col, k):
    total_relevant = group["relevant"].sum()
    if total_relevant == 0:
        return 0.0
    top_k = group.sort_values(rank_col).head(k)
    return top_k["relevant"].sum() / total_relevant


def dcg_at_k(relevances):
    relevances = np.asarray(relevances, dtype=float)
    if relevances.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, relevances.size + 2))
    return float(np.sum(relevances / discounts))


def ndcg_at_k(group, rank_col, k):
    top_k = group.sort_values(rank_col).head(k)
    actual_relevance = top_k["relevant"].values
    dcg = dcg_at_k(actual_relevance)

    ideal_relevance = np.sort(group["relevant"].values)[::-1][:k]
    idcg = dcg_at_k(ideal_relevance)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def safe_auc(data, score_col):
    y_true = data["relevant"].astype(int)
    y_score = data[score_col].astype(float)

    if y_true.nunique() < 2:
        return np.nan

    return roc_auc_score(y_true, y_score)


def mean_rank_of_relevant(group, rank_col):
    relevant_rows = group[group["relevant"] == 1]
    if len(relevant_rows) == 0:
        return np.nan
    return relevant_rows[rank_col].mean()


def mrr(group, rank_col):
    relevant_rows = group[group["relevant"] == 1].sort_values(rank_col)
    if len(relevant_rows) == 0:
        return 0.0
    first_rank = relevant_rows.iloc[0][rank_col]
    if first_rank <= 0:
        return 0.0
    return 1.0 / first_rank


In [18]:
# ===== EVALUATE MODEL =====
def evaluate_model(eval_df, model_name, score_col, rank_col):
    rows = []

    for candidate_id, group in eval_df.groupby("candidate_id"):
        row = {
            "model": model_name,
            "candidate_id": candidate_id,
            "n_jobs": int(group["job_id"].nunique()),
            "n_relevant": int(group["relevant"].sum()),
            "mean_rank_relevant": mean_rank_of_relevant(group, rank_col),
            "mrr": mrr(group, rank_col),
        }

        for k in K_VALUES:
            row[f"precision_at_{k}"] = precision_at_k(group, rank_col, k)
            row[f"recall_at_{k}"] = recall_at_k(group, rank_col, k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(group, rank_col, k)

        rows.append(row)

    candidate_metrics = pd.DataFrame(rows)

    summary = {
        "model": model_name,
        "auc": safe_auc(eval_df, score_col),
        "mean_rank_relevant": candidate_metrics["mean_rank_relevant"].mean(),
        "mean_mrr": candidate_metrics["mrr"].mean(),
    }

    for k in K_VALUES:
        summary[f"mean_precision_at_{k}"] = candidate_metrics[f"precision_at_{k}"].mean()
        summary[f"mean_recall_at_{k}"] = candidate_metrics[f"recall_at_{k}"].mean()
        summary[f"mean_ndcg_at_{k}"] = candidate_metrics[f"ndcg_at_{k}"].mean()

    return candidate_metrics, summary


In [19]:
# ===== RUN EVALUATION =====
no_tax_metrics, no_tax_summary = evaluate_model(
    eval_df=df,
    model_name="Without Taxonomy",
    score_col="hybrid_no_taxonomy_score",
    rank_col="rank_no_taxonomy",
)

tax_metrics, tax_summary = evaluate_model(
    eval_df=df,
    model_name="With Taxonomy",
    score_col="hybrid_taxonomy_score",
    rank_col="rank_taxonomy",
)

# ===== EVALUATE TAXONOMY-MATCH CASES ONLY =====
# Đánh giá riêng nhóm job phù hợp nhờ taxonomy.

if len(taxonomy_eval_df) == 0:
    print("Không có taxonomy-match cases.")
else:
    taxonomy_case_no_tax_metrics, taxonomy_case_no_tax_summary = evaluate_model(
        eval_df=taxonomy_eval_df,
        model_name="Without Taxonomy - Taxonomy Cases",
        score_col="hybrid_no_taxonomy_score",
        rank_col="rank_no_taxonomy",
    )

    taxonomy_case_tax_metrics, taxonomy_case_tax_summary = evaluate_model(
        eval_df=taxonomy_eval_df,
        model_name="With Taxonomy - Taxonomy Cases",
        score_col="hybrid_taxonomy_score",
        rank_col="rank_taxonomy",
    )

    taxonomy_case_summary = pd.DataFrame([
        taxonomy_case_no_tax_summary,
        taxonomy_case_tax_summary,
    ])

    taxonomy_case_summary

candidate_level_metrics = pd.concat(
    [no_tax_metrics, tax_metrics],
    ignore_index=True,
)

metrics_summary = pd.DataFrame([
    no_tax_summary,
    tax_summary,
])

metrics_summary


,model,auc,mean_rank_relevant,mean_mrr,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20
0,Without Taxonomy,0.965278,2.617647,0.85,0.36,0.85,0.85,0.18,0.85,0.85
1,With Taxonomy,1.000000,2.617647,0.85,0.36,0.85,0.85,0.18,0.85,0.85


In [21]:
# ===== COMPARE SUMMARY =====
display_cols = [
    "model",
    "mean_precision_at_10",
    "mean_recall_at_10",
    "mean_ndcg_at_10",
    "mean_precision_at_20",
    "mean_recall_at_20",
    "mean_ndcg_at_20",
    "auc",
    "mean_rank_relevant",
    "mean_mrr",
]

metrics_summary[display_cols]


,model,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20,auc,mean_rank_relevant,mean_mrr
0,Without Taxonomy,0.36,0.85,0.85,0.18,0.85,0.85,0.965278,2.617647,0.85
1,With Taxonomy,0.36,0.85,0.85,0.18,0.85,0.85,1.000000,2.617647,0.85


In [22]:
# ===== EXPORT =====
candidate_level_metrics.to_excel(OUTPUT_CANDIDATE_METRICS, index=False)
metrics_summary.to_excel(OUTPUT_SUMMARY, index=False)

OUTPUT_TAXONOMY_CASE_SUMMARY = DATA_DIR / "17_taxonomy_case_metrics_summary.xlsx"

if "taxonomy_case_summary" in globals():
    taxonomy_case_summary.to_excel(OUTPUT_TAXONOMY_CASE_SUMMARY, index=False)
    print("Saved ->", OUTPUT_TAXONOMY_CASE_SUMMARY)

print("Saved ->", OUTPUT_CANDIDATE_METRICS)
print("Saved ->", OUTPUT_SUMMARY)

candidate_level_metrics.head()


Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/17_taxonomy_case_metrics_summary.xlsx
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/15_candidate_level_metrics.xlsx
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/16_metrics_summary.xlsx


,model,candidate_id,n_jobs,n_relevant,mean_rank_relevant,mrr,precision_at_10,recall_at_10,ndcg_at_10,precision_at_20,recall_at_20,ndcg_at_20
0,Without Taxonomy,C001,5,5,3.0,1.0,0.5,1.0,1.0,0.25,1.0,1.0
1,Without Taxonomy,C002,5,5,3.0,1.0,0.5,1.0,1.0,0.25,1.0,1.0
2,Without Taxonomy,C003,5,5,3.0,1.0,0.5,1.0,1.0,0.25,1.0,1.0
3,Without Taxonomy,C004,5,3,2.0,1.0,0.3,1.0,1.0,0.15,1.0,1.0
4,Without Taxonomy,C005,5,0,NaN,0.0,0.0,0.0,0.0,0.00,0.0,0.0
